# 07 — multi_step at H=30, side by side with single_step

Fills the one missing cell in the horizon sweep: **15 min (H=30) has no
`multi_step` run**. `run_03_R0605-PA_L60_H30` (the 15-min row of the results
table) is from 2026-07-04, ~2.5 weeks before `multi_step` existed, so it was
never re-run under the new strategy.

That matters because a horizon-sweep table assembled today mixes strategies:
the 15-min row is recursive while 1h/1d/1w are direct. This notebook produces
the matching `multi_step` H=30 run and compares the two head to head.

Config is matched to `run_03` exactly (`lookback=60`, `horizon=30`,
`fast_mode=True`, `train_days=None`, same 5 models) so the ONLY difference is
`strategy`. Using `run_id='run_03'` means the output lands in
`run_03_R0605-PA_L60_H30_multistep`, pairing with the existing folder the same
way `run_1h_...`/`run_1h_..._multistep` already do.

## 0. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from rack_forecast import ExperimentConfig
from rack_forecast.paths import RESULTS_ROOT
from rack_forecast.pipeline import run_experiment
from rack_forecast.persistence import load_metrics

SINGLE_RUN = RESULTS_ROOT / 'run_03_R0605-PA_L60_H30'
print('single_step run exists:', SINGLE_RUN.exists())

## 1. Config — identical to run_03 except `strategy`

`direct_stride=None` lets `resolved_direct_stride()` pick automatically. At
H=30 the direct target is only ~53MB on the full training set, so it resolves
to stride=1 (every overlapping window kept) — no training examples are lost to
the memory budget here, unlike the long-horizon runs.

In [ ]:
cfg = ExperimentConfig(
    target_rack='R0605-PA',
    lookback=60,
    horizon=30,
    strategy='multi_step',
    direct_stride=None,
    models=['linear', 'xgboost', 'lstm', 'cnn1d', 'transformer'],
    fast_mode=True,
    train_days=None,
    predict_days=None,
    run_id='run_03',
)
print(cfg.run_name)
print('output folder:', cfg.run_folder)
print('already run:', cfg.run_folder.exists())

## 2. Run it

Guarded so re-executing the notebook doesn't retrain — delete the folder (or
set `FORCE = True`) to re-run from scratch.

Expect this to take a while: full training data, 5 models, 3 of them DL with
`dl_epochs=30`. XGBoost is the one to watch — `multi_step` makes it fit
`n_estimators × horizon` = 200 × 30 = 6,000 trees (fine at H=30, which is
exactly why it was dropped from the 1d/1w multistep runs where it would have
been 576k / 4M trees).

In [ ]:
FORCE = False

if cfg.run_folder.exists() and not FORCE:
    print(f'Skipping — {cfg.run_name} already exists. Set FORCE=True to re-run.')
else:
    _, metrics_df, _ = run_experiment(cfg)
    print(metrics_df)

## 3. Side-by-side comparison

In [ ]:
single = load_metrics(SINGLE_RUN)
multi = load_metrics(cfg.run_folder)

comp = single.join(multi, lsuffix='_single', rsuffix='_multi', how='outer')
# interleave so each metric's two strategies sit next to each other
comp = comp[[c for m in single.columns for c in (f'{m}_single', f'{m}_multi')]]
comp.columns = pd.MultiIndex.from_tuples(
    [(c.rsplit('_', 1)[0], c.rsplit('_', 1)[1]) for c in comp.columns],
    names=['metric', 'strategy'])
comp

### Deltas (multi_step relative to single_step)

Negative = multi_step better for MAE/RMSE/MAPE (lower is better); positive =
multi_step better for R2 (higher is better).

In [ ]:
delta = pd.DataFrame({
    'RMSE_single': single['RMSE'],
    'RMSE_multi': multi['RMSE'],
    'RMSE_%change': 100 * (multi['RMSE'] - single['RMSE']) / single['RMSE'],
    'R2_single': single['R2'],
    'R2_multi': multi['R2'],
    'R2_change': multi['R2'] - single['R2'],
}).round(4)
delta.sort_values('RMSE_%change')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = range(len(single))
w = 0.38

axes[0].bar([i - w/2 for i in x], single['RMSE'], w, label='single_step')
axes[0].bar([i + w/2 for i in x], multi.reindex(single.index)['RMSE'], w, label='multi_step')
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(single.index, rotation=30, ha='right')
axes[0].set_ylabel('RMSE'); axes[0].set_title('RMSE (lower is better)'); axes[0].legend()

axes[1].bar([i - w/2 for i in x], single['R2'], w, label='single_step')
axes[1].bar([i + w/2 for i in x], multi.reindex(single.index)['R2'], w, label='multi_step')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(single.index, rotation=30, ha='right')
axes[1].set_ylabel('R2'); axes[1].set_title('R2 (higher is better)'); axes[1].legend()

plt.suptitle('H=30 (15 min), R0605-PA — single_step vs multi_step')
plt.tight_layout()
plt.show()

## 4. Takeaway

In [ ]:
best_single = single['RMSE'].idxmin()
best_multi = multi['RMSE'].idxmin()
print(f"best single_step: {best_single}  RMSE={single.loc[best_single, 'RMSE']:.4f}")
print(f"best multi_step:  {best_multi}  RMSE={multi.loc[best_multi, 'RMSE']:.4f}")

wins = (multi.reindex(single.index)['RMSE'] < single['RMSE']).sum()
print(f'\nmulti_step beat single_step on RMSE for {wins}/{len(single)} models')

**What this is for:** with this run in place, the horizon-sweep table can be
reported as a single consistent strategy instead of mixing recursive (15 min)
with direct (1h/1d/1w). Two gaps remain even then — xgboost has no multi_step
result at 1d/1w (tree-count blowup), and cnn1d has no single_step result at 1w
(`DivergenceError` at step 1220/20160). See the horizon-sweep table in
`CLAUDE.md`.